# Решения: практикум сортировок

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import time
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден рядом с ноутбуком')


unsorted_df = pd.read_csv(_find('bank_transactions_unsorted.csv'))
by_id_df = pd.read_csv(_find('bank_transactions_sorted_by_txn_id.csv'))
by_amount_df = pd.read_csv(_find('bank_transactions_sorted_by_amount.csv'))
tiny_df = pd.read_csv(_find('bank_transactions_tiny.csv'))

unsorted_txns = list(unsorted_df[['txn_id', 'amount', 'day', 'risk_score']].itertuples(index=False, name=None))
id_txns = list(by_id_df[['txn_id', 'amount', 'day', 'risk_score']].itertuples(index=False, name=None))
amount_txns = list(by_amount_df[['txn_id', 'amount', 'day', 'risk_score']].itertuples(index=False, name=None))
id_list = [t[0] for t in id_txns]
amount_list = [t[1] for t in amount_txns]


In [ ]:
def selection_sort(nums):
    arr = nums[:]
    for i in range(len(arr)):
        m = i
        for j in range(i + 1, len(arr)):
            if arr[j] < arr[m]:
                m = j
        arr[i], arr[m] = arr[m], arr[i]
    return arr


def merge_sorted(a, b):
    i = 0
    j = 0
    out = []
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            out.append(a[i]); i += 1
        else:
            out.append(b[j]); j += 1
    out.extend(a[i:]); out.extend(b[j:])
    return out


def merge_sort(nums):
    if len(nums) <= 1:
        return nums[:]
    mid = len(nums) // 2
    return merge_sorted(merge_sort(nums[:mid]), merge_sort(nums[mid:]))


vals = [x[1] for x in unsorted_txns[:80]]
sizes = [80, 200, 500]
bench = []
for n in sizes:
    part = [x[1] for x in unsorted_txns[:n]]
    t0 = time.perf_counter(); selection_sort(part); t_sel = time.perf_counter() - t0
    t0 = time.perf_counter(); merge_sort(part); t_mer = time.perf_counter() - t0
    bench.append([n, t_sel, t_mer])
n_ok = (bench[0][1] <= bench[1][1] <= bench[2][1]) and (bench[0][2] <= bench[1][2] <= bench[2][2])
BENCH_NOTE = (
    'На маленьких объёмах разница умеренная, но с ростом n selection сортировка растёт быстрее. '
    'Для больших логов разумнее выбирать алгоритм уровня O(n log n).'
)
risks = [x[3] for x in unsorted_txns[:150]]
risk_sorted = merge_sort(risks)
part2 = [x[1] for x in unsorted_txns[:500]]
t0 = time.perf_counter(); sorted(part2); t_builtin = time.perf_counter() - t0
t0 = time.perf_counter(); merge_sort(part2); t_merge = time.perf_counter() - t0
PROD_NOTE = (
    'Встроенная сортировка в Python тщательно оптимизирована и надёжно протестирована. '
    'Свою реализацию обычно пишут для обучения и для контроля идеи, а не для боевого кода.'
)
print(bench)
print(n_ok)
print(t_builtin, t_merge)